# 04 - Live pipeline (manual run, issue #6 scenario 12)

End-to-end proof that the tested stages compose: camera frame -> YOLO
inference -> largest Person -> posture score -> dwell update -> frozen
JSON-lines on output. No new logic here; every behavior is owned and tested
in 00-03. This notebook only wires them in one kernel session.

Manual run (camera + weights required): set `BALDUR_LIVE=1` and run all
cells. Default 30 frames (~a few seconds) exercises inference, scoring,
and heartbeat; extend the range past ~10 slouched seconds to watch
`bad_posture` fire. A warmup frame is read and inferred once before the
loop so first-predict latency never skews the first dt. pytest and
headless verify run with the variable unset
and skip the live body. Cell output shows frozen event lines only.

In [ ]:
import json as _json
from pathlib import Path as _Path


def _sibling_notebook_path(name):
    for _base in (_Path.cwd(), _Path.cwd() / "notebooks"):
        _candidate = _base / name
        if _candidate.exists():
            return _candidate
    raise FileNotFoundError(name)


def _load_ns(name):
    """Exec importable cells of a sibling notebook (skips LIVE/DEMO)."""
    with open(_sibling_notebook_path(name), encoding="utf-8") as _handle:
        _notebook = _json.load(_handle)
    _namespace = {}
    for _cell in _notebook["cells"]:
        if _cell["cell_type"] != "code":
            continue
        _source = "".join(_cell["source"])
        if "# DEMO" in _source or "# LIVE" in _source:
            continue
        exec(compile(_source, name, "exec"), _namespace)
    return _namespace

In [ ]:
# LIVE: guarded end-to-end chain. Manual run with BALDUR_LIVE=1.
import os as _os
import time as _time

if _os.getenv("BALDUR_LIVE") == "1":
    _ns = {
        name: _load_ns(name)
        for name in (
            "00_env_camera.ipynb",
            "01_pose_score.ipynb",
            "02_events_jsonl.ipynb",
            "03_yolo_inference.ipynb",
        )
    }
    _cam = _ns["00_env_camera.ipynb"]
    _emit_open_error = _cam["emit_error"]
    try:
        _cap = _cam["open_camera"]()
    except Exception as _open_exc:
        _emit_open_error(
            getattr(_open_exc, "code", "CAMERA_UNAVAILABLE"),
            str(_open_exc) or type(_open_exc).__name__,
        )
        _cap = None
    if _cap is not None:
        _score_ns = _ns["01_pose_score.ipynb"]
        _events_ns = _ns["02_events_jsonl.ipynb"]
        _infer_ns = _ns["03_yolo_inference.ipynb"]
        try:
            _model, _ = _infer_ns["load_pose_model"]()
            _state = _events_ns["new_state"]()
            _warmed_up = False
            try:
                _warmup_frame = _cam["read_frame"](_cap)
                _infer_ns["infer_frame"](_model, _warmup_frame)
                _warmed_up = True
            except Exception as _warmup_exc:
                _cam["emit_error"](
                    getattr(_warmup_exc, "code", "FRAME_READ_FAILED"),
                    str(_warmup_exc) or type(_warmup_exc).__name__,
                )
            _prev = _time.time()
            for _ in range(30):
                if not _warmed_up:
                    break
                try:
                    _frame = _cam["read_frame"](_cap)
                except Exception as _read_exc:
                    _cam["emit_error"](
                        getattr(_read_exc, "code", "FRAME_READ_FAILED"),
                        str(_read_exc) or type(_read_exc).__name__,
                    )
                    break
                _now = _time.time()
                _dt, _prev = _now - _prev, _now
                _height, _width = _frame.shape[0], _frame.shape[1]

                def _score_one():
                    _persons = _infer_ns["infer_frame"](_model, _frame)
                    if not _persons:
                        return None
                    _person = _score_ns["select_largest_person"](_persons)
                    return _score_ns["posture_score"](
                        _person["xyn"],
                        _person["visibility"],
                        aspect=_width / _height,
                    )

                _frame_failed = []
                def _on_frame_error(_code, _message, _flag=_frame_failed):
                    _flag.append((_code, _message))

                for _event in _events_ns["safe_update"](_state, _score_one, _dt, on_error=_on_frame_error):
                    _events_ns["emit_event"](_event)
                if _frame_failed:
                    break
        finally:
            _cam["release_camera"](_cap)
    print("pipeline run finished")